# Image Generation — Google Colab

This notebook runs the same Flask web UI as `python app.py`, but inside
Google Colab on a GPU runtime. Supports multiple models:
- **FLUX.2 Klein** (4B / 9B) — fast, high-quality text-to-image and editing
- **Stable Diffusion 3.5 Medium** — powerful text-to-image

Steps:

1. **Runtime → Change runtime type → GPU** (an A100 / L4 / T4 with enough VRAM is recommended; the 4B model needs ~16 GB).
2. Run the cells in order. The last cell prints a clickable URL to the web UI.

You will need a Hugging Face token with access to all models:
- [FLUX.2-klein-4B](https://huggingface.co/black-forest-labs/FLUX.2-klein-4B)
- [FLUX.2-klein-9B](https://huggingface.co/black-forest-labs/FLUX.2-klein-9B)
- [Stable Diffusion 3.5 Medium](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium)

(Accept the license on each model page first.)

## 1. Clone the repository

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/Stx666Michael/image-gen-edit.git"
REPO_DIR = "/content/image-gen-edit"

if not pathlib.Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 2. Install dependencies

Colab already ships with a CUDA build of PyTorch, so we install everything
from `requirements.txt` *except* the torch packages (which would otherwise
be replaced with the CPU build).

In [ ]:
!grep -viE '^(torch|torchvision|torchaudio)([<>=!~ ]|$)' requirements.txt > /tmp/requirements-colab.txt
!pip install -q -r /tmp/requirements-colab.txt

## 3. Log in to Hugging Face

Paste a token from https://huggingface.co/settings/tokens when prompted.

In [ ]:
from huggingface_hub import login
login()

## 4. Sanity check the GPU

In [ ]:
import torch, psutil
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    free, total = torch.cuda.mem_get_info()
    print("Device:", name)
    print(f"VRAM free: {free/1e9:.1f} / {total/1e9:.1f} GB")
    if total/1e9 < 18 and "T4" in name:
        print()
        print("Note: T4 has 16 GB VRAM \u2014 use the 4B model. The 9B model")
        print("will fall back to sequential CPU offload (slow) and may still")
        print("exhaust Colab's ~12 GB RAM.")
ram = psutil.virtual_memory()
print(f"System RAM: {ram.available/1e9:.1f} / {ram.total/1e9:.1f} GB available")


## 5. Start the Flask web UI

This cell:

1. Downloads `cloudflared` (a free TCP tunnel — no signup or auth token required) and opens a public `https://*.trycloudflare.com` URL pointing at port `5000`.
2. Prints the public URL — click it to open the UI in a new tab on any device.
3. Runs the Flask server in the **foreground** (blocking). Leave the cell running for as long as you want to use the UI; press the stop button to shut both the tunnel and the server down.

The first generation will be slow because the model weights are downloaded
and loaded onto the GPU on demand.

In [ ]:
import re, subprocess, time, atexit, pathlib, urllib.request

from app import app

PORT = 5000

# --- 1. Install cloudflared (once per runtime) ---------------------------
CF_BIN = pathlib.Path("/usr/local/bin/cloudflared")
if not CF_BIN.exists():
    print("Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF_BIN,
    )
    CF_BIN.chmod(0o755)

# --- 2. Start the tunnel and capture the public URL ----------------------
log_path = "/tmp/cloudflared.log"
open(log_path, "w").close()
log_file = open(log_path, "a+")
tunnel = subprocess.Popen(
    [str(CF_BIN), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=log_file, stderr=subprocess.STDOUT,
)
atexit.register(tunnel.terminate)

public_url = None
deadline = time.time() + 30
url_re = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
while time.time() < deadline and public_url is None:
    time.sleep(1)
    with open(log_path) as f:
        m = url_re.search(f.read())
    if m:
        public_url = m.group(0)

if public_url:
    print("Open the web UI in a new tab:")
    print(f"  {public_url}")
else:
    print("Could not detect the cloudflared URL \u2014 see", log_path)
print()
print("Server is running. Stop this cell to shut it down.")

# --- 3. Run Flask in the foreground --------------------------------------
# threaded=True so progress polling doesn't block generation requests.
app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False, threaded=True)
